# MultiRocket Experiments Notebook

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV, GroupShuffleSplit
from sklearn.metrics import classification_report, f1_score, make_scorer
from sklearn.linear_model import RidgeClassifier, LogisticRegression
import itertools

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

# Ensure sktime and skopt are available
try:
    import sktime
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "sktime", "scikit-optimize", "-q"])
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True

import sys
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception as e:
    pass

import data_utils
from base_utils_qwen import (
    SequenceExtractor,
    competition_scorer,
    evaluate_holdout,
    make_competition_scorer,
    prepare_bayesian_space
)
from multi_rocket_utils import (
    FlexibleMultiRocketClassifier,
    HierarchicalMultiRocketEnsemble
)

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
MODE = "single"  # "ensemble" or "single"
target_col = "bfrb"
orientation_col = "orientation"
search_mode = "grid"  # "grid" or "bayesian"

random_state = 42
n_splits = 3
n_iter = 10
train_size = 0.2  # Using 80% for training now instead of small balanced split
error_score_constant = 0.0
verbose = 3
do_cross_val = False

# Slicing options for Single Mode
slice_by_orientation = None  # e.g., ["Seated Straight", "Standing"]
slice_by_bfrb = None         # True for BFRB only, False for non-BFRB only, None for all

do_handness = False

# Ensemble orientations split (Leave None to auto-split evenly)
l3_orientations = None 
l4_orientations = None 

if do_cross_val:
    cv_object = GroupKFold(n_splits=n_splits)
else:
    cv_object = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# Scorer selection based on target
if target_col == 'bfrb':
    scorer = competition_scorer
else:
    scorer = make_scorer(f1_score, average="macro", zero_division=0)

print(f"MODE: {MODE}")
print(f"Search mode: {search_mode}")

In [ ]:
data_root = data_utils.find_data_root()
raw_train_df = pd.read_csv(data_root / "train.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")

train_df = raw_train_df.set_index("row_id").copy(deep=True)

if do_handness:
# Handedness & Upside-down corrections
    if "handedness" in train_demo_df.columns:
        train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
        left_handed_mask = train_df["handedness"].eq(0)
        train_df.loc[left_handed_mask, "acc_x"] *= -1.0
        train_df = train_df.drop(columns=["handedness"])

    upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
    train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
    train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0

train_df["is_target"] = (train_df["sequence_type"] == "Target").astype(bool)
train_df["bfrb"] = train_df["gesture"].where(train_df["is_target"], "non_bfrb")

# Create alternative target columns
train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]

In [ ]:
import itertools
from base_utils_qwen import SequenceExtractor

# Take a small sample of X_train for fast computation
sample_df = X_train.head(2000)

# Extract extractor parameter names and values from param_space
extractor_keys = [k for k in param_space.keys() if k.startswith("extractor__")]

# Check if we're in Bayesian mode (contains skopt objects) or Grid mode (lists)
is_bayesian = search_mode == "bayesian" and SKOPT_AVAILABLE

if is_bayesian:
    print("Bayesian mode detected - sampling 5 random combinations for channel calculation\n")
    import random
    from skopt.space import Categorical, Integer, Real
    
    # For Bayesian, we'll sample random combinations
    n_samples = 5
    
    for i in range(n_samples):
        extractor_params = {}
        for key in extractor_keys:
            param_name = key.replace("extractor__", "")
            space = param_space[key]
            
            if isinstance(space, Categorical):
                value = random.choice(space.categories)
            elif isinstance(space, Integer):
                value = random.randint(space.low, space.high)
            elif isinstance(space, Real):
                if space.prior == "log-uniform":
                    value = np.exp(random.uniform(np.log(space.low), np.log(space.high)))
                else:
                    value = random.uniform(space.low, space.high)
            else:
                value = space
            
            extractor_params[param_name] = value
        
        # Ensure padding_value=0.0
        extractor_params["padding_value"] = 0.0
        
        try:
            extractor = SequenceExtractor(**extractor_params)
            extractor.fit(sample_df)
            out = extractor.transform(sample_df)
            n_channels = out['X'].shape[2]
            print(f"Sample {i+1}:")
            # Show only key params
            key_params = {k: v for k, v in extractor_params.items() if k in ['acc_modes', 'rotation_modes', 'sampling_rate', 'maxlen', 'window_size']}
            print(f"  Params: {key_params}")
            print(f"  → Channels: {n_channels}\n")
        except Exception as e:
            print(f"Failed for {extractor_params}: {e}\n")

else:
    # Grid mode - iterate over all combinations
    extractor_values = [param_space[k] for k in extractor_keys]
    total_combos = len(list(itertools.product(*extractor_values)))
    print(f"Total extractor parameter combinations: {total_combos}\n")
    
    for combo in itertools.product(*extractor_values):
        extractor_params = {k.replace("extractor__", ""): v for k, v in zip(extractor_keys, combo)}
        
        # Ensure padding_value=0.0 (required for MultiRocket)
        extractor_params["padding_value"] = 0.0
        
        try:
            extractor = SequenceExtractor(**extractor_params)
            extractor.fit(sample_df)
            out = extractor.transform(sample_df)
            n_channels = out['X'].shape[2]
            # Show only key params to avoid clutter
            key_params = {k: v for k, v in extractor_params.items() if k in ['acc_modes', 'rotation_modes', 'sampling_rate', 'maxlen', 'window_size']}
            print(f"Params: {key_params}")
            print(f"→ Channels: {n_channels}\n")
        except Exception as e:
            print(f"Failed for {extractor_params}: {e}\n")

In [ ]:
# ============================================================
# PARAMETER SPACE DEFINITION
# ============================================================
if search_mode == "bayesian" and SKOPT_AVAILABLE:
    extractor_space = {
        "extractor__acc_modes": Categorical(["smoothed|velocity|jerk", "raw|velocity|displacement|jerk"]),
        "extractor__rotation_modes": Categorical(["quaternion|angular_velocity|euler", "quaternion"]),
        "extractor__tof_modes": Categorical(["pooled_stats|sensor_stats", "sensor_stats"]),
        "extractor__thm_modes": Categorical(["centered_diff", "centered_diff|diff"]),
        "extractor__sampling_rate": Categorical([50, 100, 200]), 
        "extractor__maxlen": Categorical([120, 160]),
        "extractor__window_size": Integer(10, 40),
        "extractor__clip_value": Real(30.0, 100.0, prior="linear"),
        "extractor__interp_mode": Categorical(["linear"]),
        "extractor__motion_filter_mode": Categorical(["kalman", None]),
        "extractor__kalman_process_noise": Real(1e-4, 1e-2, prior="log-uniform"),
        "extractor__kalman_measurement_noise": Real(1e-2, 1e-1, prior="log-uniform"),
        "extractor__padding_value": Categorical([0.0]), 
    }

    # TRUE MULTIPLES OF 84 (Aeon MultiRocket requirement)
    kernels_space = Categorical([840, 1260, 1680])
    max_dilations_space = Categorical([16, 32, 64])
    pad_space = Categorical([True, False])
    use_pyfftw_space = Categorical([False, True])
    use_multivariate_space = Categorical([True])
    alpha_space = Real(1e-2, 1e4, prior="log-uniform")
    fs_space = Categorical([None, 25, 50, 75])
    channels_space = Categorical([None, 15, 20, 30])

    if MODE == "ensemble":
        model_space = {
            "classifier__base_classifier_class": Categorical([RidgeClassifier]),
            
            # Layer 1
            "classifier__l1_num_kernels": kernels_space,
            "classifier__l1_max_dilations_per_kernel": max_dilations_space,
            "classifier__l1_pad": pad_space,
            "classifier__l1_use_pyfftw": use_pyfftw_space,
            "classifier__l1_use_multivariate": use_multivariate_space,
            "classifier__l1_alpha": alpha_space, 
            "classifier__l1_feature_selection_percentile": fs_space,
            "classifier__l1_max_channels": channels_space,
            "classifier__l1_class_weight": Categorical(["balanced", None]),
            
            # Layer 2
            "classifier__l2_num_kernels": kernels_space,
            "classifier__l2_max_dilations_per_kernel": max_dilations_space,
            "classifier__l2_pad": pad_space,
            "classifier__l2_use_pyfftw": use_pyfftw_space,
            "classifier__l2_use_multivariate": use_multivariate_space,
            "classifier__l2_alpha": alpha_space, 
            "classifier__l2_feature_selection_percentile": fs_space,
            "classifier__l2_max_channels": channels_space,
            "classifier__l2_class_weight": Categorical(["balanced", None]),
            
            # Layer 3 Model 1
            "classifier__l3_1_num_kernels": kernels_space,
            "classifier__l3_1_max_dilations_per_kernel": max_dilations_space,
            "classifier__l3_1_pad": pad_space,
            "classifier__l3_1_use_pyfftw": use_pyfftw_space,
            "classifier__l3_1_use_multivariate": use_multivariate_space,
            "classifier__l3_1_alpha": alpha_space,
            "classifier__l3_1_feature_selection_percentile": fs_space,
            "classifier__l3_1_max_channels": channels_space,
            
            # Layer 3 Model 2
            "classifier__l3_2_num_kernels": kernels_space,
            "classifier__l3_2_max_dilations_per_kernel": max_dilations_space,
            "classifier__l3_2_pad": pad_space,
            "classifier__l3_2_use_pyfftw": use_pyfftw_space,
            "classifier__l3_2_use_multivariate": use_multivariate_space,
            "classifier__l3_2_alpha": alpha_space,
            "classifier__l3_2_feature_selection_percentile": fs_space,
            "classifier__l3_2_max_channels": channels_space,
            
            # Layer 3 Model 3
            "classifier__l3_3_num_kernels": kernels_space,
            "classifier__l3_3_max_dilations_per_kernel": max_dilations_space,
            "classifier__l3_3_pad": pad_space,
            "classifier__l3_3_use_pyfftw": use_pyfftw_space,
            "classifier__l3_3_use_multivariate": use_multivariate_space,
            "classifier__l3_3_alpha": alpha_space,
            "classifier__l3_3_feature_selection_percentile": fs_space,
            "classifier__l3_3_max_channels": channels_space,
            
            # Layer 3 Model 4
            "classifier__l3_4_num_kernels": kernels_space,
            "classifier__l3_4_max_dilations_per_kernel": max_dilations_space,
            "classifier__l3_4_pad": pad_space,
            "classifier__l3_4_use_pyfftw": use_pyfftw_space,
            "classifier__l3_4_use_multivariate": use_multivariate_space,
            "classifier__l3_4_alpha": alpha_space,
            "classifier__l3_4_feature_selection_percentile": fs_space,
            "classifier__l3_4_max_channels": channels_space,
            
            "classifier__l3_class_weight": Categorical(["balanced", None]),
        }
        fixed_model_params = {
            "classifier__orientation_col": [orientation_col],
            "classifier__target_col": [target_col],
            "classifier__padding_value": [0.0],
        }
    else:  # single mode
        model_space = {
            "classifier__base_classifier_class": Categorical([RidgeClassifier]),
            "classifier__num_kernels": kernels_space,
            "classifier__max_dilations_per_kernel": max_dilations_space,
            "classifier__pad": pad_space,
            "classifier__use_pyfftw": use_pyfftw_space,
            "classifier__use_multivariate": use_multivariate_space,
            "classifier__alpha": alpha_space,
            "classifier__feature_selection_percentile": fs_space,
            "classifier__max_channels": channels_space,
        }
        fixed_model_params = {
            "classifier__target_col": [target_col],
            "classifier__orientation_col": [orientation_col],
            "classifier__slice_by_orientation": [slice_by_orientation],
            "classifier__slice_by_bfrb": [slice_by_bfrb],
            "classifier__padding_value": [0.0],
            "classifier__class_weight": Categorical(["balanced", None]),
        }

    param_space = {**extractor_space, **model_space, **fixed_model_params}
    param_space = prepare_bayesian_space(param_space)

else:  # GRID SEARCH
    param_space = {
        "extractor__acc_modes": ["raw"],
        "extractor__rotation_modes": ["raw"],
        "extractor__tof_modes": ["raw"],
        "extractor__thm_modes": ["raw"],
        "extractor__sampling_rate": [20],
        "extractor__maxlen": [80],
        "extractor__window_size": [20],
        "extractor__clip_value": [50.0],
        "extractor__interp_mode": ["linear"],
        "extractor__motion_filter_mode": ["kalman"],
        "extractor__kalman_process_noise": [1e-3],
        "extractor__kalman_measurement_noise": [1e-1],
        "extractor__padding_value": [0.0],
    }
    
    # MultiRocket classifier hyperparameters for grid search
    kernals_list = [84, 168, 336, 504, 672, 840, 1260, 1680]
    max_dilations_list = [16, 32, 64]
    pad_list = [True, False]
    use_pyfftw_list = [False]  # Keep False unless you have FFTW installed
    use_multivariate_list = [True]
    alphas_list = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0]
    feature_selection_list = [None, 25, 50, 75]
    max_channels_list = [None, 15, 20, 30, 50]
    classifier_padding_value_list = [0.0]
    class_weight_list = ["balanced", None]

    if MODE == "ensemble":
        param_space.update({
            "classifier__base_classifier_class": [RidgeClassifier],
            # Layer 1
            "classifier__l1_num_kernels": kernals_list,
            "classifier__l1_max_dilations_per_kernel": max_dilations_list,
            "classifier__l1_pad": pad_list,
            "classifier__l1_use_pyfftw": use_pyfftw_list,
            "classifier__l1_use_multivariate": use_multivariate_list,
            "classifier__l1_alpha": alphas_list,
            "classifier__l1_feature_selection_percentile": feature_selection_list,
            "classifier__l1_max_channels": max_channels_list,
            "classifier__l1_class_weight": class_weight_list,
            # Layer 2
            "classifier__l2_num_kernels": kernals_list,
            "classifier__l2_max_dilations_per_kernel": max_dilations_list,
            "classifier__l2_pad": pad_list,
            "classifier__l2_use_pyfftw": use_pyfftw_list,
            "classifier__l2_use_multivariate": use_multivariate_list,
            "classifier__l2_alpha": alphas_list,
            "classifier__l2_feature_selection_percentile": feature_selection_list,
            "classifier__l2_max_channels": max_channels_list,
            "classifier__l2_class_weight": class_weight_list,
            # Layer 3 Model 1
            "classifier__l3_1_num_kernels": kernals_list,
            "classifier__l3_1_max_dilations_per_kernel": max_dilations_list,
            "classifier__l3_1_pad": pad_list,
            "classifier__l3_1_use_pyfftw": use_pyfftw_list,
            "classifier__l3_1_use_multivariate": use_multivariate_list,
            "classifier__l3_1_alpha": alphas_list,
            "classifier__l3_1_feature_selection_percentile": feature_selection_list,
            "classifier__l3_1_max_channels": max_channels_list,
            # Layer 3 Model 2
            "classifier__l3_2_num_kernels": kernals_list,
            "classifier__l3_2_max_dilations_per_kernel": max_dilations_list,
            "classifier__l3_2_pad": pad_list,
            "classifier__l3_2_use_pyfftw": use_pyfftw_list,
            "classifier__l3_2_use_multivariate": use_multivariate_list,
            "classifier__l3_2_alpha": alphas_list,
            "classifier__l3_2_feature_selection_percentile": feature_selection_list,
            "classifier__l3_2_max_channels": max_channels_list,
            # Layer 3 Model 3
            "classifier__l3_3_num_kernels": kernals_list,
            "classifier__l3_3_max_dilations_per_kernel": max_dilations_list,
            "classifier__l3_3_pad": pad_list,
            "classifier__l3_3_use_pyfftw": use_pyfftw_list,
            "classifier__l3_3_use_multivariate": use_multivariate_list,
            "classifier__l3_3_alpha": alphas_list,
            "classifier__l3_3_feature_selection_percentile": feature_selection_list,
            "classifier__l3_3_max_channels": max_channels_list,
            # Layer 3 Model 4
            "classifier__l3_4_num_kernels": kernals_list,
            "classifier__l3_4_max_dilations_per_kernel": max_dilations_list,
            "classifier__l3_4_pad": pad_list,
            "classifier__l3_4_use_pyfftw": use_pyfftw_list,
            "classifier__l3_4_use_multivariate": use_multivariate_list,
            "classifier__l3_4_alpha": alphas_list,
            "classifier__l3_4_feature_selection_percentile": feature_selection_list,
            "classifier__l3_4_max_channels": max_channels_list,
            "classifier__l3_class_weight": class_weight_list,
            "classifier__orientation_col": [orientation_col],
            "classifier__target_col": [target_col],
            "classifier__padding_value": classifier_padding_value_list,
        })
    else:  # single mode
        param_space.update({
            "classifier__base_classifier_class": [RidgeClassifier],
            "classifier__num_kernels": kernals_list,
            "classifier__max_dilations_per_kernel": max_dilations_list,
            "classifier__pad": pad_list,
            "classifier__use_pyfftw": use_pyfftw_list,
            "classifier__use_multivariate": use_multivariate_list,
            "classifier__alpha": alphas_list,
            "classifier__feature_selection_percentile": feature_selection_list,
            "classifier__max_channels": max_channels_list,
            "classifier__target_col": [target_col],
            "classifier__orientation_col": [orientation_col],
            "classifier__slice_by_orientation": [slice_by_orientation],
            "classifier__slice_by_bfrb": [slice_by_bfrb],
            "classifier__padding_value": classifier_padding_value_list,
            "classifier__class_weight": class_weight_list,
        })

print("Parameter Space Defined.")

In [ ]:
# Extract extractor parameter names and values from param_space
extractor_keys = [k for k in param_space.keys() if k.startswith("extractor__")]
extractor_values = [param_space[k] for k in extractor_keys]

# Take a small sample of X_train for fast computation
sample_df = X_train.head(2000)

print(f"Total extractor parameter combinations: {len(list(itertools.product(*extractor_values)))}\n")

for combo in itertools.product(*extractor_values):
    # Build extractor parameters dict
    extractor_params = {k.replace("extractor__", ""): v for k, v in zip(extractor_keys, combo)}
    
    # Ensure padding_value=0.0 (required for MultiRocket)
    extractor_params["padding_value"] = 0.0
    
    try:
        # Create and fit extractor
        extractor = SequenceExtractor(**extractor_params)
        extractor.fit(sample_df)
        out = extractor.transform(sample_df)
        n_channels = out['X'].shape[2]
        print(f"Params: {extractor_params}")
        print(f"→ Channels (before compression): {n_channels}\n")
    except Exception as e:
        print(f"Failed for {extractor_params}: {e}\n")

In [ ]:
pipe = Pipeline([
    # We override padding_value to 0.0 here because MultiRocket convolutions react poorly to -999.0
    ("extractor", SequenceExtractor(padding_value=0.0)), 
    ("classifier", HierarchicalMultiRocketEnsemble(padding_value=0.0) if MODE == "ensemble" else FlexibleMultiRocketClassifier(padding_value=0.0))
])

if search_mode == "bayesian" and SKOPT_AVAILABLE:
    search = BayesSearchCV(
        pipe, param_space, n_iter=n_iter, scoring=scorer,
        cv=cv_object, n_jobs=1, random_state=random_state,
        error_score=error_score_constant, verbose=verbose, return_train_score=True
    )
else:
    search = GridSearchCV(
        pipe, param_space, scoring=scorer,
        cv=cv_object, n_jobs=1, error_score=error_score_constant, 
        verbose=verbose, return_train_score=True
    )

print(f"Running {search_mode} search...")
search.fit(X_train, y_train, groups=groups)

print(f"\nBest CV score: {search.best_score_:.4f}")
print("Best parameters:")
print(search.best_params_)

best_model = search.best_estimator_

In [ ]:
if not do_cross_val and len(X_test) > 0:
    y_pred = best_model.predict(X_test)
    
    # Collapse to sequence level
    y_test_seq = y_test.drop_duplicates("sequence_id").sort_values("sequence_id")
    
    if target_col == 'bfrb':
        eval_dict = evaluate_holdout(y_test_seq, y_pred, target_col=target_col, verbose=True)
        print(f"Holdout Competition Score: {eval_dict['competition_score']:.4f}")
    else:
        print("\nClassification Report:")
        print(classification_report(y_test_seq[target_col], y_pred, zero_division=0))
        macro_f1 = f1_score(y_test_seq[target_col], y_pred, average='macro', zero_division=0)
        print(f"Macro F1 Score: {macro_f1:.4f}")
else:
    print("Holdout evaluation skipped (Cross-Val mode or empty test set).")